# Freight rate prediction - solution notebook

**Goal:** predict `posted_rate` for 12,000 loads in `validation.csv` (Nov-Dec 2025) and a 31-day December forecast for one fixed lane.

**Run order:** the cells depend on each other, so use *Run all* from the top. Place the provided files next to this notebook (`train-test.csv`, `validation.csv`, `validation-predictions-template.csv`, `december-chart-inputs.csv`, `score.py`). Outputs are written to `outputs/`.

**Short version of the approach**
1. Explore the data and fix quality problems (sign errors, missing values, corrupted rates).
2. Validate with **time-aware splits**, because validation comes entirely after training.
3. Model **log(rate per mile)** with LightGBM; dollars = prediction x distance.
4. Rates drift over time and the market columns help only some of the time, so the final prediction **averages four feature variants** to reduce worst-case error.
5. Predict, run the provided `score.py`, and produce the December chart.

## 1. Load the data and take a first look
Train has the target (`posted_rate`); validation does not. Validation dates all come *after* training dates.

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("train-test.csv", parse_dates=["date"])
val = pd.read_csv("validation.csv", parse_dates=["date"])

print("train:", train.shape, "| dates", train.date.min().date(), "to", train.date.max().date())
print("val:  ", val.shape, "| dates", val.date.min().date(), "to", val.date.max().date())
print()
print("columns in train but not in val:", set(train.columns) - set(val.columns))
train.head()

train: (48000, 14) | dates 2025-01-01 to 2025-10-31
val:   (12000, 13) | dates 2025-11-01 to 2025-12-31

columns in train but not in val: {'posted_rate'}


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


## 2. Data quality
### 2a. Types, missing values, impossible values

In [2]:
print(train.dtypes)
print()
print("missing values:")
print(train.isna().sum())
print()
print(train.describe().T[["min", "50%", "max"]].to_string())
print()
print("negative weights:", (train.weight < 0).sum())

load_id                    str
pickup                     str
delivery                   str
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment                  str
weight                 float64
date            datetime64[us]
market_index           float64
quote_signal           float64
posted_rate            float64
dtype: object

missing values:
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

                              min                  50%                  max
pickup_lat               28.35765             35.29479             44.30296
pickup_lon             -121.69849            -88.08915                -69.5
delivery_lat 

### 2b. Rate per mile
A load's total rate depends heavily on distance, so rate **per mile** is the fair way to compare loads. A few values are far outside the normal range.

In [3]:
print("weight == 47500:", (train.weight == 47500).sum())
print("distance == 70:", (train.distance == 70).sum())
print()

train["rpm"] = train.posted_rate / train.distance

print(train.rpm.describe(percentiles=[.001, .01, .05, .5, .95, .99, .999]).round(2).to_string())
print()
print("rate per mile above $5:", (train.rpm > 5).sum())
print("rate per mile below $1:", (train.rpm < 1).sum())

weight == 47500: 1191
distance == 70: 48

count    48000.00
mean         2.22
std          0.58
min          0.33
0.1%         0.44
1%           1.67
5%           1.81
50%          2.15
95%          2.74
99%          3.18
99.9%       10.07
max         14.13

rate per mile above $5: 317
rate per mile below $1: 329


### 2c. Are the extreme rates random or tied to something?
They are spread evenly across equipment and months, which suggests random label corruption and not real market behavior. Rate per mile also falls as distance grows.

In [4]:
train["month"] = train.date.dt.month
train["kind"] = np.where(train.rpm > 5, "high", np.where(train.rpm < 1, "low", "normal"))

print(train.kind.value_counts().to_string())
print()
print("share of each kind, by equipment:")
print(pd.crosstab(train.equipment, train.kind, normalize="index").round(4).to_string())
print()
print("share of each kind, by month:")
print(pd.crosstab(train.month, train.kind, normalize="index").round(4).to_string())
print()
print("median distance by kind:")
print(train.groupby("kind").distance.median().to_string())
print()
print("typical rate per mile by distance band (normal loads only):")
normal = train[train.kind == "normal"]
print(normal.groupby(pd.cut(normal.distance, [0, 150, 500, 1000, 2000, 4000])).rpm.median().round(2).to_string())

kind
normal    47354
low         329
high        317

share of each kind, by equipment:
kind         high     low  normal
equipment                        
Dry Van    0.0066  0.0062  0.9872
Flatbed    0.0067  0.0071  0.9862
Reefer     0.0066  0.0081  0.9853

share of each kind, by month:
kind     high     low  normal
month                        
1      0.0075  0.0081  0.9843
2      0.0062  0.0060  0.9878
3      0.0060  0.0073  0.9867
4      0.0058  0.0075  0.9867
5      0.0051  0.0073  0.9876
6      0.0075  0.0046  0.9879
7      0.0071  0.0059  0.9870
8      0.0067  0.0069  0.9863
9      0.0062  0.0064  0.9874
10     0.0078  0.0082  0.9839

median distance by kind:
kind
high      869.1
low       855.0
normal    954.1

typical rate per mile by distance band (normal loads only):
distance
(0, 150]        2.87
(150, 500]      2.47
(500, 1000]     2.20
(1000, 2000]    2.05
(2000, 4000]    1.91


### 2d. Negative weights, missing values, and train vs validation differences

In [5]:
# 1. Are negative weights just flipped signs? Compare their size to the positive ones.
neg = train[train.weight < 0].weight.abs()
pos = train[train.weight > 0].weight
print("negative weights (absolute value):", neg.describe()[["min", "50%", "max"]].round(0).to_dict())
print("positive weights:                 ", pos.describe()[["min", "50%", "max"]].round(0).to_dict())
print()

# 2. Are missing values spread evenly over time?
print("share missing, by month:")
print(train.groupby("month")[["weight", "market_index"]].agg(lambda s: s.isna().mean()).round(4).to_string())
print()

# 3. Does validation look like train?
train_cities = set(train.pickup) | set(train.delivery)
val_cities = set(val.pickup) | set(val.delivery)
print("cities only in validation:", sorted(val_cities - train_cities))
lane = lambda d: d.pickup + " -> " + d.delivery
print("share of val loads on lanes never seen in train:", round((~lane(val).isin(set(lane(train)))).mean(), 3))
print()
print("equipment mix:")
print(pd.concat([train.equipment.value_counts(normalize=True),
                 val.equipment.value_counts(normalize=True)], axis=1, keys=["train", "val"]).round(3).to_string())
print()
print(pd.concat([train[["market_index", "quote_signal"]].describe().T[["min", "mean", "max"]],
                 val[["market_index", "quote_signal"]].describe().T[["min", "mean", "max"]]],
                keys=["train", "val"]).round(3).to_string())

negative weights (absolute value): {'min': 5000.0, '50%': 31822.0, 'max': 47500.0}
positive weights:                  {'min': 5000.0, '50%': 31494.0, 'max': 47500.0}

share missing, by month:
       weight  market_index
month                      
1      0.0069        0.0073
2      0.0076        0.0083
3      0.0071        0.0087
4      0.0042        0.0064
5      0.0045        0.0088
6      0.0067        0.0088
7      0.0059        0.0073
8      0.0061        0.0069
9      0.0071        0.0066
10     0.0066        0.0087



cities only in validation: ['Allentown', 'Charlotte', 'Chicago', 'Jackson', 'Knoxville', 'Laredo', 'Norfolk', 'San Diego']


share of val loads on lanes never seen in train: 0.122

equipment mix:
           train    val
equipment              
Dry Van    0.567  0.565
Reefer     0.251  0.254
Flatbed    0.182  0.181



                      min   mean    max
train market_index  0.676  1.083  1.468
      quote_signal  0.692  2.062  3.610
val   market_index  0.724  0.927  1.099
      quote_signal  1.235  2.051  2.896


## 3. First split and a baseline
Forward-looking split: learn from Jan-Aug, test on Sep-Oct. The outlier rule compares each load's rate per mile to the typical value for its (equipment, distance band); the ratio table shows a clear gap, so the 0.5 / 2.0 cutoffs are safe. The baseline is the simple lookup every model has to beat.

In [6]:
train["dist_band"] = pd.cut(train.distance, [0, 150, 500, 1000, 2000, 4000])

# Forward-looking split: learn from the past, test on the future
fit_part = train[train.date < "2025-09-01"].copy()
hold = train[train.date >= "2025-09-01"].copy()
print("fit rows:", len(fit_part), "| holdout rows:", len(hold))

# Typical rate per mile, learned from the fit part only
typical = fit_part.groupby(["equipment", "dist_band"], observed=True).rpm.median().rename("typical_rpm")
fit_part = fit_part.join(typical, on=["equipment", "dist_band"])
fit_part["ratio"] = fit_part.rpm / fit_part.typical_rpm

print("\nratio of each load's rate-per-mile to the typical one:")
print(fit_part.ratio.quantile([.001, .005, .01, .02, .05, .5, .95, .98, .99, .995, .999]).round(2).to_string())

bad = (fit_part.ratio < 0.5) | (fit_part.ratio > 2.0)
print("\nflagged as corrupted:", bad.sum(), f"({bad.mean():.2%})")

# Baseline predictions on the holdout
hold = hold.join(typical, on=["equipment", "dist_band"])
hold["pred"] = hold.typical_rpm * hold.distance
hold["ratio"] = hold.rpm / hold.typical_rpm
hold_bad = (hold.ratio < 0.5) | (hold.ratio > 2.0)

mae = lambda d: (d.pred - d.posted_rate).abs().mean()
print("\nbaseline MAE, clean holdout rows:", round(mae(hold[~hold_bad]), 1))
print("baseline MAE, all holdout rows:  ", round(mae(hold), 1))

fit rows: 38477 | holdout rows: 9523



ratio of each load's rate-per-mile to the typical one:
0.001    0.21
0.005    0.34
0.010    0.86
0.020    0.88
0.050    0.90
0.500    1.00
0.950    1.11
0.980    1.15
0.990    1.19
0.995    2.95
0.999    4.57

flagged as corrupted: 533 (1.39%)

baseline MAE, clean holdout rows: 86.2
baseline MAE, all holdout rows:   141.7


## 4. First LightGBM model
Target: log(rate per mile). Loss: L1 (robust to leftover noise). Trained only on rows that passed the outlier rule. Model A uses distance, equipment and weight; model B adds the two market columns.

In [7]:
import lightgbm as lgb

# Features: fix the sign errors, encode equipment as numbers
for d in (fit_part, hold):
    d["weight_clean"] = d.weight.abs()
    d["equip"] = d.equipment.map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2})
    d["log_distance"] = np.log(d.distance)

clean_fit = fit_part[~bad]   # `bad` is the outlier flag from the last cell

feature_sets = {
    "A: distance, equipment, weight": ["distance", "log_distance", "equip", "weight_clean"],
    "B: A + market_index + quote_signal": ["distance", "log_distance", "equip", "weight_clean",
                                           "market_index", "quote_signal"],
}

for name, feats in feature_sets.items():
    model = lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05,
                              num_leaves=31, min_child_samples=100, verbose=-1, random_state=0)
    model.fit(clean_fit[feats], np.log(clean_fit.rpm))

    pred = np.exp(model.predict(hold[feats])) * hold.distance
    err = (pred - hold.posted_rate).abs()
    print(name)
    print("   MAE clean:", round(err[~hold_bad].mean(), 1),
          "| MAE all:", round(err.mean(), 1),
          "| MAPE clean:", round((err[~hold_bad] / hold.posted_rate[~hold_bad]).mean() * 100, 2), "%")
    if name.startswith("B"):
        print("   importance:", dict(zip(feats, model.booster_.feature_importance("gain").round(0))))

A: distance, equipment, weight
   MAE clean: 59.2 | MAE all: 115.0 | MAPE clean: 2.73 %


B: A + market_index + quote_signal
   MAE clean: 68.2 | MAE all: 123.9 | MAPE clean: 2.68 %
   importance: {'distance': np.float64(374033.0), 'log_distance': np.float64(0.0), 'equip': np.float64(152732.0), 'weight_clean': np.float64(58188.0), 'market_index': np.float64(81586.0), 'quote_signal': np.float64(292678.0)}


## 5. Multiple forward-looking folds
One holdout is one data point. Three expanding-window folds (train on the past, predict the next two months) show that which model wins depends on the period.

In [8]:
train["weight_clean"] = train.weight.abs()
train["equip"] = train.equipment.map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2})

FOLDS = [("2025-05-01", "2025-06-30"), ("2025-07-01", "2025-08-31"), ("2025-09-01", "2025-10-31")]

def run_fold(start, end, feats):
    fit_p = train[train.date < start]
    hold_p = train[(train.date >= start) & (train.date <= end)]

    # everything below is learned from the past only
    typical = fit_p.groupby(["equipment", "dist_band"], observed=True).rpm.median().rename("typical_rpm")
    fit_p = fit_p.join(typical, on=["equipment", "dist_band"])
    hold_p = hold_p.join(typical, on=["equipment", "dist_band"])
    fit_ratio = fit_p.rpm / fit_p.typical_rpm
    hold_ratio = hold_p.rpm / hold_p.typical_rpm
    clean_fit = fit_p[(fit_ratio >= 0.5) & (fit_ratio <= 2.0)]
    hold_ok = (hold_ratio >= 0.5) & (hold_ratio <= 2.0)

    if feats is None:                       # baseline
        pred = hold_p.typical_rpm * hold_p.distance
    else:
        m = lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05,
                              num_leaves=31, min_child_samples=100, verbose=-1, random_state=0)
        m.fit(clean_fit[feats], np.log(clean_fit.rpm))
        pred = np.exp(m.predict(hold_p[feats])) * hold_p.distance

    return (pred - hold_p.posted_rate).abs()[hold_ok].mean()

models = {"baseline": None,
          "A": ["distance", "equip", "weight_clean"],
          "B": ["distance", "equip", "weight_clean", "market_index", "quote_signal"]}

results = {name: [run_fold(s, e, f) for s, e in FOLDS] for name, f in models.items()}
table = pd.DataFrame(results, index=["May-Jun", "Jul-Aug", "Sep-Oct"]).round(1)
table.loc["mean"] = table.mean().round(1)
print("MAE on clean holdout rows, by fold:")
print(table.to_string())

print("\nmedian rate per mile by month (normal loads only):")
normal = train[train.kind == "normal"]
print(normal.groupby("month").rpm.median().round(3).to_string())

MAE on clean holdout rows, by fold:
         baseline      A     B
May-Jun     145.7  147.1  51.4
Jul-Aug      91.1   67.4  87.9
Sep-Oct      86.2   59.2  68.2
mean        107.7   91.2  69.2

median rate per mile by month (normal loads only):
month
1     2.029
2     2.061
3     2.140
4     2.147
5     2.191
6     2.247
7     2.187
8     2.121
9     2.160
10    2.164


## 6. What do the market columns measure?
`market_index` is essentially one number per day (little variation within a day) and tracks the rate level. `quote_signal` has a **U-shaped** relationship with rates, so its plain correlation understates its information.

In [9]:
typical_all = train.groupby(["equipment", "dist_band"], observed=True).rpm.transform("median")
train["ratio"] = train.rpm / typical_all
eda = train[(train.ratio >= 0.5) & (train.ratio <= 2.0)].copy()   # drop corrupted rows
eda["lvl"] = np.log(eda.ratio) * 100                               # rate level, in %

print("row-level correlation with the rate level:")
print(eda[["lvl", "market_index", "quote_signal"]].corr().loc["lvl"].round(3).to_string())

daily = eda.groupby("date").agg(lvl=("lvl", "mean"),
                                mi_day=("market_index", "median"),
                                qs_day=("quote_signal", "mean"))
print("\nday-level correlation with the rate level:")
print(daily.corr().loc["lvl"].round(3).to_string())

print("\nrate level (% vs typical) by day-level market_index quintile:")
print(daily.groupby(pd.qcut(daily.mi_day, 5), observed=True).lvl.mean().round(2).to_string())

print("\nrate level (%) by row-level quote_signal bucket:")
print(eda.groupby(pd.qcut(eda.quote_signal, 8), observed=True).lvl.mean().round(2).to_string())

print("\nhow much does market_index vary within one day vs across days?")
print("within-day std (average):", round(eda.groupby("date").market_index.std().mean(), 3),
      "| std of the daily medians:", round(daily.mi_day.std(), 3))

row-level correlation with the rate level:
lvl             1.000
market_index    0.381
quote_signal    0.072

day-level correlation with the rate level:
lvl       1.000
mi_day    0.697
qs_day    0.076

rate level (% vs typical) by day-level market_index quintile:
mi_day
(0.768, 0.927]   -2.40
(0.927, 1.009]   -1.82
(1.009, 1.118]   -0.85
(1.118, 1.252]    1.49
(1.252, 1.401]    3.52

rate level (%) by row-level quote_signal bucket:
quote_signal
(0.691, 1.772]    1.88
(1.772, 1.891]   -1.50
(1.891, 1.979]   -1.20
(1.979, 2.056]   -0.64
(2.056, 2.134]   -0.56
(2.134, 2.222]   -0.60
(2.222, 2.36]    -0.43
(2.36, 3.61]      3.09

how much does market_index vary within one day vs across days?
within-day std (average): 0.025 | std of the daily medians: 0.167


## 7. A day-level market feature
`mi_day` is the median `market_index` of each day. It uses feature columns only (never the label), so it can be computed for validation days too. The monthly table shows the link between `market_index` and the rate level is **not stable over time** (January vs September).

In [10]:
train["month"] = train.date.dt.month
train["mi_day"] = train.groupby("date").market_index.transform("median")
print("missing market_index:", train.market_index.isna().sum(), "| missing mi_day:", train.mi_day.isna().sum())

base = ["distance", "equip", "weight_clean"]
models = {"A: base": base,
          "B: + market_index + quote_signal (row)": base + ["market_index", "quote_signal"],
          "C: + mi_day": base + ["mi_day"],
          "D: + mi_day + quote_signal": base + ["mi_day", "quote_signal"]}

results = {name: [run_fold(s, e, f) for s, e in FOLDS] for name, f in models.items()}
table = pd.DataFrame(results, index=["May-Jun", "Jul-Aug", "Sep-Oct"]).round(1)
table.loc["mean"] = table.mean().round(1)
print("MAE on clean holdout rows:")
print(table.T.to_string())

eda["mi_day"] = eda.date.map(train.groupby("date").market_index.median())
monthly = eda.groupby("month").agg(avg_mi_day=("mi_day", "mean"),
                                   rate_level_pct=("lvl", "mean"),
                                   median_rpm=("rpm", "median"))
print("\nby month:")
print(monthly.round(2).to_string())

missing market_index: 374 | missing mi_day: 0


MAE on clean holdout rows:
                                        May-Jun  Jul-Aug  Sep-Oct  mean
A: base                                   147.1     67.4     59.2  91.2
B: + market_index + quote_signal (row)     51.4     87.9     68.2  69.2
C: + mi_day                                74.7     68.5    118.2  87.1
D: + mi_day + quote_signal                 52.4     91.2     69.8  71.1

by month:
       avg_mi_day  rate_level_pct  median_rpm
month                                        
1            0.93           -5.41        2.03
2            1.00           -3.96        2.06
3            1.07           -0.38        2.14
4            1.21            0.12        2.15
5            1.30            2.37        2.19
6            1.28            4.81        2.25
7            1.20            1.95        2.19
8            0.98           -1.13        2.12
9            0.89            0.56        2.16
10           0.96            0.80        2.16


## 8. Leave-one-month-out check
Holding out one month at a time (10 situations instead of 3) shows where each model breaks. Models using `quote_signal` do worst on August, the only month whose `quote_signal` profile resembles November-December.

In [11]:
train["month"] = train.date.dt.month

def run_month(m, feats):
    fit_p = train[train.month != m]
    hold_p = train[train.month == m]
    typical = fit_p.groupby(["equipment", "dist_band"], observed=True).rpm.median().rename("typical_rpm")
    fit_p = fit_p.join(typical, on=["equipment", "dist_band"])
    hold_p = hold_p.join(typical, on=["equipment", "dist_band"])
    fr = fit_p.rpm / fit_p.typical_rpm
    hr = hold_p.rpm / hold_p.typical_rpm
    clean_fit = fit_p[(fr >= 0.5) & (fr <= 2.0)]
    hold_ok = (hr >= 0.5) & (hr <= 2.0)

    mdl = lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05, num_leaves=31,
                            min_child_samples=100, verbose=-1, random_state=0)
    mdl.fit(clean_fit[feats], np.log(clean_fit.rpm))
    pred = np.exp(mdl.predict(hold_p[feats])) * hold_p.distance
    y = hold_p.posted_rate
    return (pred - y).abs()[hold_ok].mean(), (pred / y - 1)[hold_ok].mean() * 100

rows = []
for m in range(1, 11):
    for name, f in {"A": base, "B": base + ["market_index", "quote_signal"],
                    "C": base + ["mi_day"], "D": base + ["mi_day", "quote_signal"]}.items():
        mae, bias = run_month(m, f)
        rows.append({"month": m, "model": name, "MAE": mae, "bias": bias})
r = pd.DataFrame(rows)

print("MAE by held-out month:")
print(r.pivot(index="month", columns="model", values="MAE").round(0).to_string())
print("\nbias (% over/under-predicting):")
print(r.pivot(index="month", columns="model", values="bias").round(1).to_string())
print("\nsummary:")
print(r.groupby("model").agg(mean_MAE=("MAE", "mean"), worst_MAE=("MAE", "max")).round(1).to_string())

a = train.groupby("month").agg(qs_mean=("quote_signal", "mean"), qs_std=("quote_signal", "std"), mi_median=("market_index", "median"))
b = val.groupby(val.date.dt.month).agg(qs_mean=("quote_signal", "mean"), qs_std=("quote_signal", "std"), mi_median=("market_index", "median"))
print("\nmonthly profile (months 11 and 12 are the validation set):")
print(pd.concat([a, b]).round(3).to_string())

MAE by held-out month:
model      A      B      C      D
month                            
1      140.0   59.0  115.0   66.0
2      109.0   41.0   85.0   43.0
3       66.0   33.0   65.0   36.0
4       61.0   44.0   74.0   45.0
5       78.0   33.0   60.0   34.0
6      131.0   81.0   83.0   86.0
7       75.0   36.0   60.0   37.0
8       60.0  117.0   57.0  115.0
9       61.0   61.0  113.0   63.0
10      57.0   73.0   92.0   74.0

bias (% over/under-predicting):
model    A    B    C    D
month                    
1      6.2  2.3  5.0  2.7
2      4.6  1.4  3.3  1.5
3      0.5 -0.0  0.0  0.0
4      0.1  1.0  2.4  1.0
5     -2.5  0.2  1.1  0.2
6     -5.0 -1.7 -2.6 -1.7
7     -2.0 -0.2 -0.4 -0.3
8      1.4 -1.3 -0.7 -1.4
9     -0.4 -1.9 -4.3 -2.0
10    -0.8 -2.2 -3.4 -2.3

summary:
       mean_MAE  worst_MAE
model                     
A          83.8      140.4
B          57.9      117.1
C          80.6      115.2
D          59.9      115.0

monthly profile (months 11 and 12 are the validatio

## 9. Averaging the four models
Because no single model is safe everywhere, the final prediction averages models A-D (in log space). This cell scores single models and the average under both validation schemes. The average was chosen after seeing these results, so its numbers are slightly optimistic; the claim is that it is *safer*, not proven best.

In [12]:
train["month"] = train.date.dt.month
blend_feature_sets = {"A": base,
                      "B": base + ["market_index", "quote_signal"],
                      "C": base + ["mi_day"],
                      "D": base + ["mi_day", "quote_signal"]}

def score_split(fit_p, hold_p):
    typical = fit_p.groupby(["equipment", "dist_band"], observed=True).rpm.median().rename("typical_rpm")
    fit_p = fit_p.join(typical, on=["equipment", "dist_band"])
    hold_p = hold_p.join(typical, on=["equipment", "dist_band"])
    fr = fit_p.rpm / fit_p.typical_rpm
    hr = hold_p.rpm / hold_p.typical_rpm
    clean_fit = fit_p[(fr >= 0.5) & (fr <= 2.0)]
    ok = (hr >= 0.5) & (hr <= 2.0)

    logs = {}
    for name, feats in blend_feature_sets.items():
        m = lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05, num_leaves=31,
                              min_child_samples=100, verbose=-1, random_state=0)
        m.fit(clean_fit[feats], np.log(clean_fit.rpm))
        logs[name] = m.predict(hold_p[feats])
    logs["average of all four"] = np.mean(list(logs.values()), axis=0)

    return {name: (np.exp(lg) * hold_p.distance - hold_p.posted_rate).abs()[ok].mean()
            for name, lg in logs.items()}

lomo = pd.DataFrame({m: score_split(train[train.month != m], train[train.month == m])
                     for m in range(1, 11)}).T
fwd = pd.DataFrame({s: score_split(train[train.date < s], train[(train.date >= s) & (train.date <= e)])
                    for s, e in FOLDS}).T

summary = pd.DataFrame({"LOMO mean": lomo.mean(), "LOMO worst": lomo.max(),
                        "forward mean": fwd.mean(), "forward worst": fwd.max()}).round(1)
print("MAE in dollars on clean holdout rows (lower is better):")
print(summary.to_string())

MAE in dollars on clean holdout rows (lower is better):
                     LOMO mean  LOMO worst  forward mean  forward worst
A                         83.8       140.4          91.2          147.1
B                         57.9       117.1          69.2           87.9
C                         80.6       115.2          87.1          118.2
D                         59.9       115.0          71.1           91.2
average of all four       63.1        93.1          68.9           72.3


## 10. Final fit and predictions
Train the four models on all clean training rows, average them, and predict the validation loads and the December rows. December has no market columns, so the day's market values are taken from the validation loads on the same date.

In [13]:
import os
os.makedirs("outputs", exist_ok=True)

# ---- 1. Features for the validation loads (same recipe as train) ----
val["weight_clean"] = val.weight.abs()
val["equip"] = val.equipment.map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2})
val["mi_day"] = val.groupby("date").market_index.transform("median")

# ---- 2. Features for the 31 December rows ----
dec = pd.read_csv("december-chart-inputs.csv")
dec_dates = pd.to_datetime(dec["date"])
dec_feat = pd.DataFrame({
    "distance": dec["distance"].astype(float),
    "equip": dec["equipment"].map({"Dry Van": 0, "Flatbed": 1, "Reefer": 2}),
    "weight_clean": dec["weight"].astype(float),
    "mi_day": dec_dates.map(val.groupby("date").market_index.median()),
    "quote_signal": dec_dates.map(val.groupby("date").quote_signal.mean()),
})
dec_feat["market_index"] = dec_feat["mi_day"]
assert dec_feat.notna().all().all(), "some December dates have no market data"

# ---- 3. Drop corrupted training rows (same rule, using ALL training data) ----
typical = train.groupby(["equipment", "dist_band"], observed=True).rpm.transform("median")
ratio = train.rpm / typical
clean_train = train[(ratio >= 0.5) & (ratio <= 2.0)]
print("training rows used:", len(clean_train), "of", len(train))

# ---- 4. Fit the four models, average their predictions (in log space) ----
feature_sets = {"A": base,
                "B": base + ["market_index", "quote_signal"],
                "C": base + ["mi_day"],
                "D": base + ["mi_day", "quote_signal"]}
val_log, dec_log = [], []
for name, feats in feature_sets.items():
    m = lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05, num_leaves=31,
                          min_child_samples=100, verbose=-1, random_state=0)
    m.fit(clean_train[feats], np.log(clean_train.rpm))
    val_log.append(m.predict(val[feats]))
    dec_log.append(m.predict(dec_feat[feats]))

val["predicted_rate"] = (np.exp(np.mean(val_log, axis=0)) * val.distance).round(2)
dec["predicted_rate"] = (np.exp(np.mean(dec_log, axis=0)) * dec_feat.distance).round(2)

# ---- 5. Save in the format the scorer wants ----
template = pd.read_csv("validation-predictions-template.csv")
out = template[["load_id"]].merge(val[["load_id", "predicted_rate"]], on="load_id", how="left")
assert out.predicted_rate.notna().all() and (out.predicted_rate > 0).all() and len(out) == 12000
out.to_csv("outputs/validation_predictions.csv", index=False)
dec.to_csv("outputs/december_chart_inputs.csv", index=False)

print("validation predictions:", out.predicted_rate.describe().round(0)[["min", "50%", "max"]].to_dict())
print("December: min", dec.predicted_rate.min(), "| mean", round(dec.predicted_rate.mean(), 1), "| max", dec.predicted_rate.max())

training rows used: 47323 of 48000


validation predictions: {'min': 179.0, '50%': 2006.0, 'max': 6740.0}
December: min 798.53 | mean 805.5 | max 813.78


## 11. Run the provided scorer
Validates both files and creates the December chart (`outputs/scorer_results/candidate_december.png`).

In [14]:
!python score.py --predictions outputs/validation_predictions.csv --december-predictions outputs/december_chart_inputs.csv --output-dir outputs/scorer_results

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: outputs/scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.


## Limitations
- The overall rate level drifts month to month and the link between `market_index` and rates is unstable, so the **Nov-Dec level is the largest uncertainty**.
- December has no training data and the chart input contains only a date, so the December line is nearly flat; its small day-to-day variation is weak evidence.
- Eight validation cities never appear in training (about 12% of loads are on unseen lanes); they are handled only through distance, equipment and market features.
- The hidden evaluation metric is unknown. If hidden labels contain the same ~1.4% corruption, any submission's error will be dominated by it.